# Database SQLite

## Imports

In [ ]:
import sqlite3
import pandas as pd
import os

## Création de la base de données.

In [ ]:
# Fonction création et connexion à la base de données.


def create_database_connection():

    # Chemin du répertoire pour la base de donnée.
    db_folder_path = os.path.join("..", "data", "database")

    # Vérifier et créer le répertoire de destination s'il n'existe pas.
    if not os.path.exists(db_folder_path):
        os.makedirs(db_folder_path)

    db_path = os.path.join("..", "data", "database", "idfm.db")
    return sqlite3.connect(db_path)

## Création de tables et importation des données.

In [ ]:
# Fonction pour créer les tables et importer les données vers la base.


def insert_data_from_csv(csv_file, table_name, conn):
    df = pd.read_csv(csv_file)
    df.to_sql(table_name, conn, if_exists="replace", index=False)

In [ ]:
def main():
    # Créer une connexion à la base de données.
    conn = create_database_connection()

    try:
        # Insérer les données dans les tables.
        insert_data_from_csv(
            os.path.join("..", "data", "processed", "stations.csv"), 
            "stations", 
            conn,
        )
        insert_data_from_csv(
            os.path.join("..", "data", "processed", "validations.csv"),
            "validations",
            conn,
        )
        insert_data_from_csv(
            os.path.join("..", "data", "processed", "validations_ligne.csv"),
            "validations_ligne",
            conn,
        )
        
        insert_data_from_csv(
            os.path.join("..", "data", "processed", "validations_fusion.csv"),
            "validations_fusion",
            conn,
        )
    except Exception as e:
        print(f"An error occurred: {e}")
    finally:
        # S'assurer que la connexion est fermée.
        conn.close()

In [ ]:
if __name__ == "__main__":
    main()

## Tests.

In [ ]:
# Connexion à la base de données.
db_path = os.path.join("..", "data", "database", "idfm.db")
conn = sqlite3.connect(db_path)
cur = conn.cursor()

In [ ]:
# Test requête table stations.
cur.execute(
    "SELECT id_ref_zdc, nom_zdc, res_com, exploitant, latitude, longitude FROM stations WHERE nom_zdc = 'Châtelet'"
)

print(cur.fetchone())

In [ ]:
# Test requête table validations.
cur.execute("SELECT * FROM validations WHERE id_zdc = '71434'")

print(cur.fetchone())

In [ ]:
# Test requête table validations_ligne.
cur.execute("SELECT Ligne, somme_nb_vald FROM validations_ligne WHERE Ligne = 'RER A'")

print(cur.fetchone())

In [ ]:
# Test requête table validations_fusion.
cur.execute("SELECT jour, id_zdc, nom_zdc, res_com, categorie_titre, nb_vald, latitude, longitude FROM validations_fusion WHERE nom_zdc = 'Trocadéro' AND jour = '2025-08-15' AND categorie_titre = 'Forfaits courts'")

print(cur.fetchone())

In [ ]:
# Test requête avec jointure.
cur.execute(
    "SELECT validations.jour, stations.nom_zdc, stations.latitude, stations.longitude, validations.categorie_titre, validations.nb_vald FROM validations JOIN stations ON validations.id_zdc = stations.id_ref_zdc"
)

print(cur.fetchone())

In [ ]:
# Fermer la connexion.
conn.close()

In [ ]:
# Test pandas
# Connexion à la base de données SQLite, table validations_fusion.
db_path = os.path.join("..", "data", "database", "idfm.db")
with sqlite3.connect(db_path) as conn :
    # Requête SQL.
    query = "SELECT * FROM validations_fusion"
    # Charger les données dans un DataFrame.
    df = pd.read_sql_query(query, conn)
df.head()

In [ ]:
df.shape

In [ ]:
# Test pandas
# Connexion à la base de données SQLite, table validations_fusion.
db_path = os.path.join("..", "data", "database", "idfm.db")
with sqlite3.connect(db_path) as conn :
    # Requête SQL.
    query = "SELECT * FROM validations_ligne"
    # Charger les données dans un DataFrame.
    lignes_df = pd.read_sql_query(query, conn)
lignes_df.head(10)